# 05 — Exploratory Analysis & Business Insights

## Objective
Integrate sentiment data with actual World Cup match results and answer the core business question:

> **Does a match result cause a measurable shift in public sentiment toward a team?**

## Method
1. Fetch fixtures and results from football-data.org.
2. For each comment, determine if it falls in a 24h-before or 24h-after window around a match involving the mentioned team.
3. Compare average sentiment pre vs. post using the Mann-Whitney U test.

## Business value
This is the project's *raison d'être*: quantifying the causal link between on-field events and public perception. A federation or sponsor can use this to anticipate brand sentiment shifts and plan communications.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.results_api import (
    fetch_matches, matches_to_dataframe, assign_matches_to_comments,
    compute_sentiment_shift, get_team_result
)
from src.utils import load_dataframe
import pandas as pd
import matplotlib.pyplot as plt

### Load full dataset (with topics + sentiment)

In [ ]:
df = load_dataframe(str(project_root / 'data' / 'processed' / 'topic_ner.parquet'))
print(f"Loaded {len(df)} comments")
display(df[['text_clean', 'sentiment_label', 'topic_label', 'teams']].head(3))

### Fetch match data

In [ ]:
raw_matches = fetch_matches(use_cache=True)
matches_df = matches_to_dataframe(raw_matches)
print(f"Fetched {len(matches_df)} matches")
display(matches_df.head())

### Map comments to matches

In [ ]:
df_enriched = assign_matches_to_comments(df, matches_df)
matched = df_enriched[df_enriched['match_id'].notna()]
print(f"Comments matched to a match: {len(matched)} ({len(matched)/len(df_enriched):.1%})")
display(matched[['text_clean', 'teams', 'team_result', 'pre_post']].head())

### Sentiment shift analysis

In [ ]:
shift_df = compute_sentiment_shift(df_enriched)
if not shift_df.empty:
    display(shift_df)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for i, metric in enumerate(['pre_pos_pct', 'pre_neg_pct']):
        post_metric = metric.replace('pre_', 'post_')
        for _, row in shift_df.iterrows():
            axes[i].plot(
                ['Before', 'After'],
                [row[metric], row[post_metric]],
                marker='o',
                label=f"{row['team']} ({row['result']})"
            )
        axes[i].set_title('Positive %' if 'pos' in metric else 'Negative %')
        axes[i].legend()
    
    plt.tight_layout()
    plt.show()
    
    sig = shift_df[shift_df['significant'] == True]
    if not sig.empty:
        print("\n⚠️  Statistically significant shifts (p < 0.05):")
        display(sig[['team', 'result', 'pre_pos_pct', 'post_pos_pct', 'p_value']])
    else:
        print("\nNo statistically significant shifts detected yet.")

### Sentiment evolution over time

In [ ]:
if 'published_at' in df.columns:
    df['date'] = pd.to_datetime(df['published_at'], utc=True).dt.date
    
    daily_sent = df.groupby('date')['sentiment_label'].value_counts(normalize=True).unstack()
    daily_sent.plot(kind='area', stacked=True, figsize=(12, 5), 
                    title='Daily Sentiment Distribution')
    plt.ylabel('Proportion')
    plt.tight_layout()
    plt.show()

### Summary of findings
| Question | Finding | Evidence |
|----------|---------|----------|
| Do wins affect sentiment? | *(to be determined after data collection)* | p-value |
| Which team has most volatile sentiment? | *(to be determined)* | Stdev of daily sentiment |
| What topic generates most negative comments? | *(to be determined)* | Topic × sentiment crosstab |

---
**Return to**: [Dashboard](../dashboard/app.py)